<!--
File: notebooks/01_EDA.ipynb
Purpose: Exploratory data analysis for the BCCD dataset.
Author: Aman Verma (23bcs012), Ankit Kashyap (23bcs017), Abhishek Chauhan (23bcs003)
Affiliation: School of Computer science and Engineering, Shri Mata Vaishno Devi University
Contact: info@smvdu.ac.in
-->

# Exploratory Data Analysis
This notebook summarizes the BCCD dataset splits and sample images.

In [ ]:
from pathlib import Path
import sys
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'configs').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.dataset import load_samples, stratified_split_indices, count_by_class

config_path = PROJECT_ROOT / 'configs' / 'config.yaml'
with open(config_path, 'r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

class_names = config['data']['class_names']
train_dir = PROJECT_ROOT / config['paths']['train_dir']
test_dir = PROJECT_ROOT / config['paths']['test_dir']
seed = int(config['project']['seed'])
val_split = float(config['data']['val_split'])

In [ ]:
train_samples, train_labels = load_samples(train_dir, class_names)
train_indices, val_indices = stratified_split_indices(train_labels, val_split, seed)

train_subset = [train_samples[i] for i in train_indices]
val_subset = [train_samples[i] for i in val_indices]
test_samples, _ = load_samples(test_dir, class_names)

train_counts = count_by_class(train_subset, class_names)
val_counts = count_by_class(val_subset, class_names)
test_counts = count_by_class(test_samples, class_names)

train_counts, val_counts, test_counts

In [ ]:
stats_df = pd.DataFrame({
    'Train': train_counts,
    'Val': val_counts,
    'Test': test_counts,
})
stats_df['Total'] = stats_df.sum(axis=1)
stats_df.loc['Total'] = stats_df.sum(axis=0)
stats_df

In [ ]:
plot_df = stats_df.drop(index='Total')
ax = plot_df[['Train', 'Val', 'Test']].plot(kind='bar', figsize=(8, 5))
ax.set_title('Class Distribution by Split')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
from PIL import Image

samples_by_class = {name: [] for name in class_names}
for path, label in train_samples:
    class_name = class_names[label]
    if len(samples_by_class[class_name]) < 2:
        samples_by_class[class_name].append(path)
    if all(len(paths) == 2 for paths in samples_by_class.values()):
        break

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()
index = 0
for class_name in class_names:
    for img_path in samples_by_class[class_name]:
        image = Image.open(img_path).convert('RGB')
        axes[index].imshow(image)
        axes[index].set_title(class_name)
        axes[index].axis('off')
        index += 1
plt.suptitle('Sample Images (2 per class)')
plt.tight_layout()
plt.show()